In [9]:
# === SECTION 1: IMPORTS ET CONFIGURATION ===

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import psycopg2
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mutual_info_score
import warnings
warnings.filterwarnings('ignore')

# Configuration de l'affichage pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Style des graphiques
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [10]:
# === CONFIGURATION POSTGRESQL ===

DB_CONFIG = {
    "host": "172.18.0.1",
    "database": "postgres",
    "user": "postgres",
    "password": "postgres",
    "port": 5441,
}

# Créer la chaîne de connexion SQLAlchemy
connection_string = f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

try:
    # Créer le moteur de base de données
    engine = create_engine(connection_string)
    
    # Tester la connexion
    with engine.connect() as connection:
        result = connection.execute(text("SELECT version();"))
        db_version = result.fetchone()[0]
        print("✓ Connexion PostgreSQL établie avec succès")
        print(f"  Version: {db_version.split(',')[0]}")
        
except Exception as e:
    print(f"✗ Erreur de connexion: {e}")
    raise

✓ Connexion PostgreSQL établie avec succès
  Version: PostgreSQL 18.1 (Debian 18.1-1.pgdg13+2) on x86_64-pc-linux-gnu


## Section 1: Configuration et Connexion PostgreSQL

Importer les bibliothèques nécessaires et établir la connexion à la base PostgreSQL.

# Construction des Features Utilisateurs pour le Clustering

## Objectif
Extraire les données de PostgreSQL et construire une table optimisée `user_features` contenant des features numériques pertinentes pour le clustering ultérieur, avec un indice de clusterabilité indiquant la richesse du profil utilisateur.

## Workflow
1. Connexion DB et inspection
2. Nettoyage des données
3. Parsing hiérarchique des catégories
4. Feature engineering au niveau utilisateur
5. Calcul de l'indice de clusterabilité
6. Création et insertion dans PostgreSQL

In [11]:
# === SECTION 10: VÉRIFICATION FINALE ===

print("=" * 80)
print("VÉRIFICATION FINALE - CONTENU DE user_features")
print("=" * 80)

try:
    # Charger les données finales depuis la base
    df_final_in_db = pd.read_sql_table('user_features', engine)
    
    print(f"\n✓ Table 'user_features' chargée depuis PostgreSQL")
    print(f"  Dimensions: {df_final_in_db.shape}")
    
    # Aperçu des premières lignes
    print(f"\n--- Premières lignes ---")
    print(df_final_in_db.head(10))
    
    # Aperçu des dernières lignes
    print(f"\n--- Dernières lignes ---")
    print(df_final_in_db.tail(10))
    
    # Statistiques
    print(f"\n--- Statistiques des features ---")
    print(df_final_in_db.describe().round(4))
    
    # Distribution du clusterability_index
    if 'clusterability_index' in df_final_in_db.columns:
        print(f"\n--- Distribution du Clusterability Index ---")
        print(df_final_in_db['clusterability_index'].describe().round(2))
        
        # Visualiser la distribution
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Histogramme
        axes[0].hist(df_final_in_db['clusterability_index'], bins=30, edgecolor='black', color='steelblue')
        axes[0].set_xlabel('Clusterability Index', fontsize=12)
        axes[0].set_ylabel('Nombre d\'utilisateurs', fontsize=12)
        axes[0].set_title('Distribution du Clusterability Index', fontsize=13, fontweight='bold')
        axes[0].grid(axis='y', alpha=0.3)
        
        # Boîte à moustaches
        axes[1].boxplot(df_final_in_db['clusterability_index'], vert=True)
        axes[1].set_ylabel('Clusterability Index', fontsize=12)
        axes[1].set_title('Boîte à Moustaches - Clusterability Index', fontsize=13, fontweight='bold')
        axes[1].grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    # Liste des colonnes finales
    print(f"\n" + "=" * 80)
    print("COLONNES FINALES RETENUES POUR LE CLUSTERING")
    print("=" * 80)
    
    columns_list = df_final_in_db.columns.tolist()
    print(f"\nTotal: {len(columns_list)} colonnes\n")
    
    for i, col in enumerate(columns_list, 1):
        dtype = str(df_final_in_db[col].dtype)
        non_null = df_final_in_db[col].notna().sum()
        null_count = df_final_in_db[col].isna().sum()
        
        print(f"{i:2d}. {col:35s} | Type: {dtype:15s} | Non-null: {non_null:6d} | Null: {null_count:6d}")
    
    print(f"\n✓ Notebook complété avec succès")
    print(f"\nRésumé:")
    print(f"  - {len(df_final_in_db)} utilisateurs traités")
    print(f"  - {len(columns_list)} features créées")
    print(f"  - Table 'user_features' prête pour le clustering")
    
except Exception as e:
    print(f"\n✗ Erreur lors de la vérification finale: {e}")

VÉRIFICATION FINALE - CONTENU DE user_features

✗ Erreur lors de la vérification finale: Table user_features not found


## Section 10: Vérification Finale et Aperçu des Résultats

Charger le contenu final de `user_features` depuis la base et afficher les résultats.

In [12]:
# === SECTION 9: INSERTION/UPSERT DES DONNÉES ===

if df_final_features is not None:
    print("=" * 80)
    print("INSERTION/UPSERT DES DONNÉES")
    print("=" * 80)
    
    try:
        # Option 1: Utiliser replace pour écraser la table (plus simple)
        df_final_features.to_sql(
            'user_features',
            engine,
            if_exists='replace',  # Replace la table entière
            index=False
        )
        
        print(f"\n✓ Données insérées avec succès (table remplacée)")
        print(f"  {len(df_final_features)} utilisateurs avec leurs features")
        
    except Exception as e:
        print(f"\n⚠ Erreur lors de l'insertion avec to_sql: {e}")
        print("\nTentative alternative avec DELETE + INSERT...")
        
        try:
            # Alternative: supprimer puis insérer
            from sqlalchemy import text as sql_text
            
            with engine.begin() as connection:
                # Supprimer la table existante
                connection.execute(sql_text("DROP TABLE IF EXISTS user_features"))
            
            # Réinsérer les données
            df_final_features.to_sql(
                'user_features',
                engine,
                if_exists='replace',
                index=False
            )
            
            print(f"\n✓ Données insérées avec succès (méthode alternative)")
            print(f"  {len(df_final_features)} utilisateurs avec leurs features")
                
        except Exception as e2:
            print(f"\n✗ Erreur lors de l'insertion alternative: {e2}")
            print("\n💡 Conseil: Vérifiez que:")
            print("  - La base de données est accessible")
            print("  - Vous avez les permissions d'écriture")
            print("  - La table user_features n'est pas verrouillée")

NameError: name 'df_final_features' is not defined

## Section 9: Insertion et Upsert des Données

Implémenter une fonction d'insertion/upsert (ON CONFLICT DO UPDATE) pour insérer les données dans user_features.

In [ ]:
# === SECTION 8: CRÉATION TABLE SQL ===

def generate_create_table_sql(feature_columns):
    """
    Génère le DDL SQL pour créer la table user_features.
    
    Args:
        feature_columns (list): Liste des colonnes de features
        
    Returns:
        str: Requête SQL CREATE TABLE
    """
    
    sql_lines = [
        "CREATE TABLE IF NOT EXISTS user_features (",
        "    user_id BIGINT PRIMARY KEY,",
    ]
    
    # Ajouter les colonnes en excluant user_id qui est déjà défini
    for col in feature_columns:
        if col != 'user_id':
            sql_lines.append(f"    {col} NUMERIC(12, 4) /* Feature: {col} */,")
    
    # Retirer la virgule du dernier élément et fermer
    if len(sql_lines) > 2:
        sql_lines[-1] = sql_lines[-1].rstrip(',')
    
    sql_lines.extend([
        "    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,",
        "    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP",
        ");",
        "",
        "-- Créer un index sur created_at pour les requêtes temporelles",
        "CREATE INDEX IF NOT EXISTS idx_user_features_created_at",
        "ON user_features(created_at);",
    ])
    
    return '\n'.join(sql_lines)

if df_final_features is not None:
    print("=" * 80)
    print("CRÉATION DE LA TABLE user_features")
    print("=" * 80)
    
    # Générer le SQL
    feature_columns = df_final_features.columns.tolist()
    create_table_sql = generate_create_table_sql(feature_columns)
    
    print("\nDDL SQL généré:")
    print("-" * 80)
    print(create_table_sql)
    print("-" * 80)
    
    # Exécuter la création
    try:
        with engine.connect() as connection:
            # Exécuter le CREATE TABLE
            connection.execute(text(create_table_sql))
            connection.commit()
            
            print("\n✓ Table 'user_features' créée ou vérifiée avec succès")
            
    except Exception as e:
        print(f"\n✗ Erreur lors de la création: {e}")
        print("  (La table existe peut-être déjà, ce n'est pas un problème)")

## Section 8: Création de la Table user_features en SQL

Générer et exécuter le DDL SQL pour créer la table `user_features` si elle n'existe pas.

In [ ]:
# === ANALYSE DE CORRÉLATION ===

if df_final_features is not None:
    print("\n" + "=" * 80)
    print("ANALYSE DE CORRÉLATION")
    print("=" * 80)
    
    # Calculer la matrice de corrélation
    # Exclure user_id de la corrélation
    features_to_correlate = [f for f in df_final_features.columns if f != 'user_id']
    corr_matrix = df_final_features[features_to_correlate].corr()
    
    print(f"\nTop 10 paires de features fortement corrélées:")
    
    # Créer une liste de paires de corrélations
    corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            feature1 = corr_matrix.columns[i]
            feature2 = corr_matrix.columns[j]
            corr_value = corr_matrix.iloc[i, j]
            corr_pairs.append((feature1, feature2, abs(corr_value), corr_value))
    
    # Trier par valeur absolue de corrélation
    corr_pairs.sort(key=lambda x: x[2], reverse=True)
    
    for i, (f1, f2, abs_corr, corr) in enumerate(corr_pairs[:10], 1):
        print(f"  {i:2d}. {f1:30s} <-> {f2:30s}: {corr:7.3f}")
    
    # Visualiser la matrice de corrélation
    if len(features_to_correlate) > 1:
        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0,
                    square=True, ax=ax, cbar_kws={'label': 'Corrélation'})
        plt.title('Matrice de Corrélation des Features', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print("\n✓ Matrice de corrélation affichée")

In [ ]:
# === SECTION 7: SÉLECTION ET VALIDATION DES FEATURES ===

if df_user_features is not None:
    print("=" * 80)
    print("SÉLECTION ET VALIDATION DES FEATURES")
    print("=" * 80)
    
    # Définir les features finales compactes
    # Features comportementales, monétaires, diversité + top catégories + clusterability
    core_features = [
        'user_id',
        'nb_events',
        'nb_sessions',
        'nb_purchases',
        'purchase_rate',
        'session_intensity',
        'total_spent',
        'avg_price_viewed',
        'nb_distinct_cat_lvl1',
        'category_entropy',
        'clusterability_index'
    ]
    
    # Ajouter les ratios de catégories (qui ont été créés)
    category_ratio_features = [col for col in df_user_features.columns if col.startswith('ratio_cat_')]
    
    final_features = core_features + category_ratio_features
    
    # Filtrer pour les features qui existent réellement
    final_features = [f for f in final_features if f in df_user_features.columns]
    
    # Créer le dataframe final avec les features sélectionnées
    df_final_features = df_user_features[final_features].copy()
    
    print(f"\n✓ Features sélectionnées: {len(final_features)} colonnes")
    print(f"\nColonnes retenues:")
    for i, col in enumerate(final_features, 1):
        print(f"  {i:2d}. {col}")
    
    # Nettoyage des NaN et inf
    print(f"\n--- Gestion des valeurs manquantes et aberrantes ---")
    
    print(f"\nValeurs NaN avant nettoyage:")
    nan_counts = df_final_features.isnull().sum()
    if nan_counts.sum() > 0:
        print(nan_counts[nan_counts > 0])
    else:
        print("  Aucune valeur NaN")
    
    # Remplacer les NaN par 0 (pratique métier: pas de données = 0)
    df_final_features = df_final_features.fillna(0)
    
    # Gérer les valeurs infinies
    print(f"\nValeurs infinies avant nettoyage:")
    inf_mask = np.isinf(df_final_features.select_dtypes(include=[np.number])).sum()
    if inf_mask.sum() > 0:
        print(inf_mask[inf_mask > 0])
        df_final_features = df_final_features.replace([np.inf, -np.inf], 0)
        print("  ✓ Valeurs infinies remplacées par 0")
    else:
        print("  Aucune valeur infinie")
    
    # Validation de la matrice
    print(f"\n✓ Validation complétée")
    print(f"\nTableau des features finales:")
    print(df_final_features.describe())

## Section 7: Sélection et Validation des Features

Sélectionner le set compact de features finales et valider la qualité de la matrice de features.

In [ ]:
# === SECTION 6: CLUSTERABILITY INDEX ===

def calculate_clusterability_index(df_user_features, df_events):
    """
    Calcule un indice de clusterabilité (0-100) indiquant si le profil utilisateur
    est suffisamment riche pour être clusterisé.
    
    L'indice est basé sur:
    1. Nombre minimum d'événements (richesse comportementale)
    2. Diversité catégorique
    3. Montant dépensé (comportement monétaire)
    4. Variation des prix
    5. Complétude des données
    
    Args:
        df_user_features (DataFrame): Features utilisateurs
        df_events (DataFrame): Données événements
        
    Returns:
        Series: Indice de clusterabilité (0-100) pour chaque utilisateur
    """
    
    scores = pd.DataFrame(index=df_user_features.index)
    
    # Critère 1: Nombre d'événements (min 5, max 100)
    # Les utilisateurs avec peu d'événements sont moins clusterisables
    min_events_threshold = df_events.groupby('user_id').size().quantile(0.25)
    max_events_threshold = df_events.groupby('user_id').size().quantile(0.95)
    
    score_events = np.clip(
        (df_user_features['nb_events'] - min_events_threshold) / 
        (max_events_threshold - min_events_threshold) * 40,
        0, 40
    )
    scores['score_events'] = score_events
    
    # Critère 2: Diversité catégorique (entropie)
    # Entropie plus élevée = meilleure clusterabilité
    max_entropy = np.log2(df_events['cat_lvl1'].nunique())
    score_entropy = (df_user_features['category_entropy'] / max_entropy) * 25
    score_entropy = np.clip(score_entropy, 0, 25)
    scores['score_entropy'] = score_entropy
    
    # Critère 3: Comportement monétaire (total_spent)
    # Les utilisateurs qui achètent sont plus clusterisables
    has_spent = df_user_features['total_spent'] > 0
    score_monetary = has_spent.astype(int) * 20
    scores['score_monetary'] = score_monetary
    
    # Critère 4: Nombre de sessions
    # Plus de sessions = profil plus stable et clusterisable
    max_sessions = df_user_features['nb_sessions'].max()
    min_sessions = df_user_features['nb_sessions'].min()
    score_sessions = np.clip(
        (df_user_features['nb_sessions'] - min_sessions) / 
        (max_sessions - min_sessions) * 15
        if max_sessions > min_sessions else 15,
        0, 15
    )
    scores['score_sessions'] = score_sessions
    
    # Score final: somme pondérée (0-100)
    clusterability_index = (
        scores['score_events'] +
        scores['score_entropy'] +
        scores['score_monetary'] +
        scores['score_sessions']
    )
    
    # Arrondir et clipper
    clusterability_index = np.clip(clusterability_index, 0, 100).round(2)
    
    return clusterability_index

# Calculer et ajouter l'indice de clusterabilité
if df_events is not None and df_user_features is not None:
    print("=" * 80)
    print("CALCUL DU CLUSTERABILITY INDEX")
    print("=" * 80)
    
    df_user_features['clusterability_index'] = calculate_clusterability_index(
        df_user_features, df_events
    )
    
    print(f"\n✓ Clusterability index calculé")
    print(f"\nStatistiques du clusterability_index:")
    print(df_user_features['clusterability_index'].describe())
    
    print(f"\nDistribution par catégorie:")
    print(f"  < 20: {(df_user_features['clusterability_index'] < 20).sum()} utilisateurs")
    print(f"  20-40: {((df_user_features['clusterability_index'] >= 20) & (df_user_features['clusterability_index'] < 40)).sum()} utilisateurs")
    print(f"  40-60: {((df_user_features['clusterability_index'] >= 40) & (df_user_features['clusterability_index'] < 60)).sum()} utilisateurs")
    print(f"  60-80: {((df_user_features['clusterability_index'] >= 60) & (df_user_features['clusterability_index'] < 80)).sum()} utilisateurs")
    print(f"  >= 80: {(df_user_features['clusterability_index'] >= 80).sum()} utilisateurs")

## Section 6: Calcul de l'Indice de Clusterabilité

Développer une fonction pour calculer un `clusterability_index` (0-100) basé sur la richesse du profil utilisateur.

In [ ]:
# === ADDITION DES RATIOS DE CATÉGORIES TOP ===

def add_category_ratios(df_user_features, df_events, n_top_categories=5):
    """
    Ajoute les ratios des top catégories niveau 1 pour chaque utilisateur.
    
    Args:
        df_user_features (DataFrame): Features utilisateurs
        df_events (DataFrame): Données événements avec cat_lvl1
        n_top_categories (int): Nombre de top catégories à inclure (3-5)
        
    Returns:
        DataFrame: Features étendues avec ratios de catégories
    """
    
    # Identifier les top catégories au niveau du dataset
    top_categories = df_events['cat_lvl1'].value_counts().head(n_top_categories).index.tolist()
    print(f"\nTop {n_top_categories} catégories (niveau 1) du dataset:")
    for i, cat in enumerate(top_categories, 1):
        count = (df_events['cat_lvl1'] == cat).sum()
        pct = 100 * count / len(df_events)
        print(f"  {i}. {cat}: {count} événements ({pct:.2f}%)")
    
    # Créer les colonnes de ratio pour chaque top catégorie
    for cat in top_categories:
        col_name = f'ratio_cat_{cat}'
        
        # Calculer pour chaque utilisateur le ratio d'événements dans cette catégorie
        ratios = []
        for user_id in df_user_features['user_id']:
            user_events = df_events[df_events['user_id'] == user_id]
            user_events_in_cat = (user_events['cat_lvl1'] == cat).sum()
            ratio = user_events_in_cat / len(user_events) if len(user_events) > 0 else 0
            ratios.append(ratio)
        
        df_user_features[col_name] = ratios
    
    return df_user_features, top_categories

# Ajouter les ratios de catégories
if df_events is not None and df_user_features is not None:
    print("\n" + "=" * 80)
    print("RATIOS DE CATÉGORIES TOP")
    print("=" * 80)
    
    df_user_features, top_categories = add_category_ratios(df_user_features, df_events, n_top_categories=5)
    
    print(f"\n✓ Ratios de catégories ajoutés")
    print(f"\nAperçu avec ratios:")
    print(df_user_features.head())

In [ ]:
# === SECTION 5: FEATURE ENGINEERING ===

def calculate_entropy(series):
    """
    Calcule l'entropie d'une série (diversité des catégories).
    Entropie haute = diversité élevée, Entropie basse = diversité faible
    """
    value_counts = series.value_counts()
    if len(value_counts) == 0:
        return 0.0
    probabilities = value_counts / len(series)
    entropy = -np.sum(probabilities * np.log2(probabilities + 1e-10))
    return entropy

def engineer_user_features(df_events):
    """
    Construit les features au niveau utilisateur.
    
    Crée pour chaque utilisateur:
    - Features comportementales: nb_events, nb_sessions, session_intensity
    - Features monétaires: nb_purchases, purchase_rate, total_spent, avg_price_viewed
    - Features de diversité: nb_distinct_cat_lvl1, category_entropy
    - Features de ratios de catégories top
    
    Args:
        df_events (DataFrame): Données d'événements nettoyées
        
    Returns:
        DataFrame: Features au niveau utilisateur
    """
    
    # Identifier les colonnes clés
    has_price = any(col in df_events.columns for col in 
                    ['price', 'amount', 'unit_price', 'product_price'])
    price_col = next((col for col in df_events.columns if 
                     any(p in col.lower() for p in ['price', 'amount'])), None)
    
    has_qty = any(col in df_events.columns for col in 
                  ['qty', 'quantity', 'quantity_purchased'])
    qty_col = next((col for col in df_events.columns if 
                   any(q in col.lower() for q in ['qty', 'quantity'])), None)
    
    has_event_type = 'event_type' in df_events.columns
    
    print(f"\nColonnes détectées:")
    print(f"  - Colonne price: {price_col}")
    print(f"  - Colonne quantity: {qty_col}")
    print(f"  - Event type disponible: {has_event_type}")
    
    # Identifier les acheteurs (event_type='purchase' ou price > 0)
    if has_event_type:
        purchases_mask = df_events['event_type'].str.lower() == 'purchase'
    elif price_col:
        purchases_mask = df_events[price_col] > 0
    else:
        purchases_mask = pd.Series([False] * len(df_events))
    
    # Créer une colonne de session (grouper par user et time window)
    # Simplification: utiliser une session par jour si date disponible
    if 'event_time' in df_events.columns:
        df_events['event_date'] = pd.to_datetime(df_events['event_time']).dt.date
        df_events['session_id'] = df_events.groupby(['user_id', 'event_date']).ngroup()
    else:
        df_events['session_id'] = df_events['user_id']
    
    # Grouper par utilisateur
    user_features = []
    
    for user_id, user_group in df_events.groupby('user_id'):
        features = {'user_id': user_id}
        
        # FEATURES COMPORTEMENTALES
        features['nb_events'] = len(user_group)
        features['nb_sessions'] = user_group['session_id'].nunique()
        features['session_intensity'] = features['nb_events'] / max(features['nb_sessions'], 1)
        
        # FEATURES MONÉTAIRES
        user_purchases = user_group[purchases_mask]
        features['nb_purchases'] = len(user_purchases)
        features['purchase_rate'] = features['nb_purchases'] / features['nb_events'] if features['nb_events'] > 0 else 0
        
        if price_col and has_price:
            features['total_spent'] = user_group[price_col].sum()
            features['avg_price_viewed'] = user_group[price_col].mean()
        else:
            features['total_spent'] = 0.0
            features['avg_price_viewed'] = 0.0
        
        # FEATURES DE DIVERSITÉ
        features['nb_distinct_cat_lvl1'] = user_group['cat_lvl1'].nunique()
        features['category_entropy'] = calculate_entropy(user_group['cat_lvl1'].dropna())
        
        user_features.append(features)
    
    df_user_features = pd.DataFrame(user_features)
    
    return df_user_features, df_events

# Exécuter le feature engineering
if df_events is not None:
    print("=" * 80)
    print("FEATURE ENGINEERING AU NIVEAU UTILISATEUR")
    print("=" * 80)
    
    df_user_features, df_events = engineer_user_features(df_events)
    
    print(f"\n✓ Features créées pour {len(df_user_features)} utilisateurs")
    print(f"\nAperçu des features:")
    print(df_user_features.head())

## Section 5: Feature Engineering au Niveau Utilisateur

Construire les features numériques pertinentes pour chaque utilisateur incluant comportement, monétaire et diversité.

In [ ]:
# === SECTION 4: PARSING HIERARCHIE CATEGORY_CODE ===

def parse_category_hierarchy(category_code):
    """
    Parse la hiérarchie du category_code délimitée par '.'
    
    Exemple: 'electronics.phones.smartphones' -> 
             {'cat_lvl1': 'electronics', 'cat_lvl2': 'phones', 'cat_lvl3': 'smartphones'}
    
    Args:
        category_code (str): Code de catégorie avec hiérarchie
        
    Returns:
        dict: Dictionnaire avec les niveaux hiérarchiques
    """
    if pd.isna(category_code) or not isinstance(category_code, str):
        return {'cat_lvl1': None, 'cat_lvl2': None, 'cat_lvl3': None}
    
    parts = category_code.split('.')
    
    return {
        'cat_lvl1': parts[0] if len(parts) > 0 else None,
        'cat_lvl2': parts[1] if len(parts) > 1 else None,
        'cat_lvl3': parts[2] if len(parts) > 2 else None,
    }

# Appliquer le parsing
if df_events is not None and 'category_code' in df_events.columns:
    print("=" * 80)
    print("PARSING DE LA HIÉRARCHIE CATEGORY_CODE")
    print("=" * 80)
    
    # Parser les catégories
    category_hierarchy = df_events['category_code'].apply(parse_category_hierarchy)
    
    # Créer les colonnes de hiérarchie
    df_events['cat_lvl1'] = category_hierarchy.apply(lambda x: x['cat_lvl1'])
    df_events['cat_lvl2'] = category_hierarchy.apply(lambda x: x['cat_lvl2'])
    df_events['cat_lvl3'] = category_hierarchy.apply(lambda x: x['cat_lvl3'])
    
    # Analyse de la distribution des niveaux
    print(f"\n✓ Parsing complété")
    print(f"\nDistribution CAT_LVL1 (niveau principal):")
    print(f"  Nombre de catégories uniques: {df_events['cat_lvl1'].nunique()}")
    print(f"\n  Top 10:")
    print(df_events['cat_lvl1'].value_counts().head(10))
    
    print(f"\nExemples de hiérarchie complète:")
    for idx in df_events[df_events['category_code'].notna()].index[:5]:
        print(f"  {df_events.loc[idx, 'category_code']} -> " +
              f"L1: {df_events.loc[idx, 'cat_lvl1']}, " +
              f"L2: {df_events.loc[idx, 'cat_lvl2']}, " +
              f"L3: {df_events.loc[idx, 'cat_lvl3']}")

## Section 4: Parsing de la Hiérarchie Category_Code

Créer une fonction pour parser la hiérarchie du `category_code` en niveaux hiérarchiques (délimitée par '.').

In [ ]:
# === SECTION 3: NETTOYAGE ET PRÉPARATION ===

if df_events is not None:
    print("=" * 80)
    print("NETTOYAGE DES DONNÉES")
    print("=" * 80)
    
    # Enregistrer les compteurs de nettoyage
    initial_rows = len(df_events)
    
    # Supprimer les lignes avec user_id manquant
    if 'user_id' in df_events.columns:
        df_events = df_events.dropna(subset=['user_id'])
        print(f"\n✓ User_id manquants supprimés: {initial_rows - len(df_events)} lignes")
    
    # Supprimer les lignes avec category_code manquant
    if 'category_code' in df_events.columns:
        before = len(df_events)
        df_events = df_events.dropna(subset=['category_code'])
        print(f"✓ Category_code manquants supprimés: {before - len(df_events)} lignes")
    
    # Convertir user_id en int si possible
    if 'user_id' in df_events.columns:
        df_events['user_id'] = pd.to_numeric(df_events['user_id'], errors='coerce')
        df_events = df_events.dropna(subset=['user_id'])
        df_events['user_id'] = df_events['user_id'].astype('int64')
    
    # Nettoyer les espaces dans category_code
    if 'category_code' in df_events.columns:
        df_events['category_code'] = df_events['category_code'].str.strip()
    
    # Récupérer les noms de colonnes pour identifier price, qty, event_type
    available_cols = df_events.columns.tolist()
    print(f"\n✓ Données après nettoyage: {len(df_events)} lignes")
    print(f"  Colonnes: {available_cols}")
    
    # Convertir les colonnes numériques supposées (price, quantity)
    numeric_patterns = ['price', 'amount', 'qty', 'quantity', 'quantity_purchased']
    for col in df_events.columns:
        if any(pattern in col.lower() for pattern in numeric_patterns):
            df_events[col] = pd.to_numeric(df_events[col], errors='coerce')
    
    print("\n✓ Nettoyage terminé")

## Section 3: Nettoyage et Préparation des Données

Nettoyer les données en supprimant les valeurs manquantes critiques, en standardisant les formats et en gérant les doublons.

In [ ]:
# === ANALYSE DÉTAILLÉE DES CATÉGORIES ===

if df_events is not None and 'category_code' in df_events.columns:
    print("\n" + "=" * 80)
    print("ANALYSE DES CATEGORIES")
    print("=" * 80)
    
    # Analyse des category_code
    print(f"\nNombre de catégories uniques: {df_events['category_code'].nunique()}")
    print(f"\nTop 20 catégories:")
    print(df_events['category_code'].value_counts().head(20))
    
    # Afficher quelques exemples de category_code
    print(f"\nExemples de category_code:")
    sample_categories = df_events[df_events['category_code'].notna()]['category_code'].drop_duplicates().head(10)
    for cat in sample_categories:
        print(f"  - {cat}")

# Afficher les colonnes disponibles
if df_events is not None:
    print("\n" + "=" * 80)
    print("COLONNES DISPONIBLES DANS all_event")
    print("=" * 80)
    print(df_events.columns.tolist())

In [ ]:
# === SECTION 2: INSPECTION DES TABLES ===

print("=" * 80)
print("CHARGEMENT DES TABLES SOURCE")
print("=" * 80)

# Charger la table all_event
print("\n1. Chargement de 'all_event'...")
try:
    df_events = pd.read_sql_table('all_event', engine)
    print(f"   ✓ {len(df_events)} lignes chargées")
except Exception as e:
    print(f"   ✗ Erreur: {e}")
    df_events = None

# Charger la table user_event si elle existe
print("\n2. Chargement de 'user_event'...")
try:
    df_user_events = pd.read_sql_table('user_event', engine)
    print(f"   ✓ {len(df_user_events)} lignes chargées")
except Exception as e:
    print(f"   ⚠ Table non disponible ou erreur: {e}")
    df_user_events = None

if df_events is not None:
    print("\n" + "=" * 80)
    print("INFORMATION SUR all_event")
    print("=" * 80)
    print(f"\nDimensions: {df_events.shape}")
    print(f"\nTypes de colonnes:")
    print(df_events.dtypes)
    print(f"\nValeurs manquantes:")
    print(df_events.isnull().sum())
    print(f"\nAperçu des données:")
    print(df_events.head())

## Section 2: Inspection des Tables Source

Charger les tables `all_event` et `user_event` et analyser leur structure et contenu.

## Prochaines Étapes

### Points Clés du Workflow:

1. **Nettoyage des données**: Les événements sans user_id ou category_code sont supprimés
2. **Parsing hiérarchique**: À partir du category_code, extraction de 3 niveaux (cat_lvl1, cat_lvl2, cat_lvl3)
3. **Features comportementales**: nb_events, nb_sessions, session_intensity
4. **Features monétaires**: total_spent, purchase_rate, avg_price_viewed
5. **Features de diversité**: nb_distinct_cat_lvl1, category_entropy
6. **Ratios de catégories**: 5 ratios basés sur les top catégories du dataset
7. **Clusterability Index**: Indicateur composite (0-100) de qualité du profil utilisateur

### Table user_features:
- **Clé primaire**: user_id
- **Colonnes de métadonnées**: created_at, updated_at
- **Total features**: ~14-16 colonnes numériques pertinentes pour le clustering

### Utilisation pour le Clustering:
```python
# Exemple pour charger la table pour le clustering
import pandas as pd
from sqlalchemy import create_engine

df = pd.read_sql_table('user_features', engine)

# Filtrer les utilisateurs clusterisables (index >= 50)
df_clusterizable = df[df['clusterability_index'] >= 50]

# Features numériques (excluant user_id)
X = df_clusterizable.drop(['user_id', 'created_at', 'updated_at'], axis=1)

# Utiliser X pour le clustering (KMeans, DBSCAN, etc.)
```